# needle-fp4: FP4 training and inference experiments on Colab

**Before running:** Runtime > Change runtime type > **GPU**. Then Runtime > Run all.
Open this notebook from GitHub (File > Open notebook > GitHub) rather than from the Drive clone:
the setup cell resets the Drive clone to match GitHub, which discards edits made inside it.
If the repo is private, add a GitHub token first (see the setup cell).

What it does, fastest and most important first:
1. Clones (or updates) the repo into `MyDrive/10714/fp4-needle`, so results survive the session, and builds needle.
2. Runs all tests in `tests/fp4`, including the CUDA kernel tests that need a GPU.
3. **Stage 4, real 4-bit weights:** GEMV kernel benchmark (time, bandwidth, memory) and MNIST inference from packed weights.
4. **MNIST MLP** trained with each FP4 configuration (SGD), plus the gradient-error study.
5. **LSTM language model** on Penn Treebank with each FP4 configuration.
6. Optional: the MNIST grid again with Adam.

**How long it takes:** with `QUICK = True` roughly 15 to 25 minutes. The full run (`QUICK = False`) trains
every configuration with 3 seeds and can take several hours; keep the tab open. Each finished run is
saved to Drive immediately, and `--only NAME` reruns single configs if the session drops.

In [ ]:
# Which branch to run (a dev/<name>-<topic> branch to try unmerged work).
BRANCH = "main"
DEVICE = "cuda"   # switch to "cpu" if the GPU turns out slower (see the timing cell)

# QUICK = True: 1 seed, 1 epoch, small data, to check that everything works (minutes).
# QUICK = False: the full experiments (3 seeds, 3 epochs, all of MNIST, 2000 lines of PTB).
QUICK = True
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = 1 if QUICK else 3
SUBSET = "--train-subset 10000" if QUICK else ""
LSTM_LINES = 500 if QUICK else 2000

In [ ]:
import base64, subprocess
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/10714

# Private repo: create a GitHub token with read access to this repo, then in Colab click the key
# icon (Secrets) > Add new secret, name it GITHUB_TOKEN and allow notebook access. The token is
# passed to git as a header for each command and is never written to Drive.
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
basic = base64.b64encode(f"x-access-token:{token}".encode()).decode() if token else ""
AUTH = f"-c http.extraheader='Authorization: Basic {basic}'" if token else ""
print("Using a GitHub token from Colab Secrets" if token else "No GITHUB_TOKEN secret: cloning as public")

!test -d fp4-needle || git {AUTH} clone https://github.com/shreya-v7/fp4-needle.git
%cd /content/drive/MyDrive/10714/fp4-needle
# Match GitHub exactly. Colab autosaves into the Drive copy of this notebook, which would
# otherwise make `git pull` refuse to update and leave old code in place.
# (Only tracked files are reset; results/, data/ and build/ are git-ignored and kept.)
!git {AUTH} fetch origin && git checkout -f {BRANCH} && git reset --hard origin/{BRANCH}
import subprocess
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
remote = subprocess.check_output(['git', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
assert head == remote, f"Checkout is at {head[:7]}, but origin/{BRANCH} is {remote[:7]}; see the git output above."
!git log --oneline -3

In [ ]:
!pip3 install -q pybind11
!make clean && make
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
sys.path.append('python'); sys.path.append('apps')
import needle as ndl
print('CUDA backend built:', ndl.cuda().enabled())
if DEVICE == 'cuda' and not ndl.cuda().enabled():
    raise RuntimeError("No CUDA backend: Runtime > Change runtime type > GPU, then Runtime > Run all again.")

## Tests
All of `tests/fp4` should pass. On a GPU the `cuda()` variants run too: they check that the
new `floor` / `abs` kernels work and that FP4 fake quantization on the GPU is bit-identical
to the NumPy reference.

In [ ]:
!python3 -m pytest tests/fp4 -q

## MNIST data
Copies MNIST from a homework folder on Drive (`MyDrive/10714/hw2/data`, `hw1` or `hw0`) if there is one,
otherwise downloads it from the course's hw0 repository. Both are byte-identical; the MD5 check confirms it.


In [ ]:
import os, hashlib, shutil, urllib.request

# MNIST files and the MD5 of the copy our homeworks used (identical to the course repo).
MNIST = {"train-images-idx3-ubyte.gz": "0cb5796cd8e7c7bffef03edbdb01fac5",
         "train-labels-idx1-ubyte.gz": "23e0cdb926d142c2d470a3641a19d1ff",
         "t10k-images-idx3-ubyte.gz":  "3d4db374c734e4f92d346e063bf6e41a",
         "t10k-labels-idx1-ubyte.gz":  "0407fd84edb9a00bbad73c377ff95de9"}
os.makedirs("data", exist_ok=True)
for name, md5 in MNIST.items():
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        # Prefer a homework copy on Drive; otherwise download from the course's hw0 repo.
        local = [p for p in (f"../{hw}/data/{name}" for hw in ("hw2", "hw1", "hw0")) if os.path.exists(p)]
        if local:
            shutil.copy(local[0], dst)
        else:
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/dlsyscourse/hw0/main/data/{name}", dst)
    assert hashlib.md5(open(dst, "rb").read()).hexdigest() == md5, f"{dst} is corrupted; delete it and rerun"
    print("ok", name)


## Timing check
Needle's CUDA matmul uses 4x4 tiles and fake quantization launches many small kernels, so the
GPU is not guaranteed to beat the CPU. This times a few steps on each device.

In [ ]:
import time
import numpy as np
import fp4_experiment as E
from needle.fp4.config import QuantConfig

train, _ = E.load_mnist('data', 1280)
for dev_name in ['cpu'] + (['cuda'] if ndl.cuda().enabled() else []):
    dev = getattr(ndl, dev_name)()
    for cfg in [QuantConfig(), QuantConfig(fprop=True, dgrad=True, wgrad=True, stochastic=True)]:
        np.random.seed(0)
        model = E.make_mlp(cfg, 256, dev)
        opt = ndl.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
        loss_fn = ndl.nn.SoftmaxLoss()
        start, steps = time.time(), 0
        for batch in ndl.data.DataLoader(train, batch_size=128, shuffle=True):
            X, y = E.to_device(batch, dev)
            opt.reset_grad(); loss = loss_fn(model(X), y); loss.backward(); opt.step()
            float(loss.numpy())  # forces the GPU to finish this step
            steps += 1
        print(f"{dev_name:5s} {cfg.name:22s} {(time.time() - start) / steps * 1000:6.0f} ms/step")

## Stage 4: real 4-bit weights (memory and speed)

The training experiments below use *fake* quantization: values are rounded to FP4 but stored and
multiplied in FP32, so they show what FP4 does to **accuracy** and are slower, not faster. Here the
weights are really stored in 4 bits (two codes per byte + one E4M3 scale byte per 16 values) and our
CUDA kernel multiplies directly from that packed form. This is what saves **memory** and **time**
when generating text, where every step is a matrix-vector product (GEMV).

In [ ]:
!python3 benchmarks/bench_gemv.py --device cuda --sizes 1024 2048 4096 8192 --out results/bench_gemv.csv
from IPython.display import Image, Markdown, display
display(Image('results/bench_gemv.png'))

In [ ]:
# MNIST MLP run from packed 4-bit weights: memory per layer, test accuracy, batch-1 latency
!python3 apps/fp4_inference.py --device {DEVICE} {'--epochs 1 --train-subset 10000' if QUICK else ''}

## MNIST MLP: the project grid plus single-matmul ablations (SGD)
10 configurations x `SEEDS`. Each run uses the same seed for every config, so runs differ only by FP4.

It also runs `apps/fp4_grad_error.py`, which measures how far each FP4 matmul moves the
weight gradients from FP32 on a fixed batch, and whether stochastic rounding is unbiased.

In [ ]:
for seed in SEEDS:
    !python3 apps/fp4_experiment.py --configs all --device {DEVICE} --epochs {EPOCHS} --seed {seed} {SUBSET} --out results/colab_sgd_seed{seed}.csv
# Gradient error of each FP4 matmul vs FP32 at 3 points of training (a few minutes)
!python3 apps/fp4_grad_error.py --device {DEVICE} --out results/colab_grad_error.csv
!python3 apps/plot_fp4.py results/colab_sgd_seed*.csv --out-dir results/colab_sgd --grad-error results/colab_grad_error.csv

In [ ]:
from IPython.display import Image, Markdown, display
display(Image('results/colab_sgd/mnist_grid.png'))
display(Image('results/colab_sgd/mnist_ablation.png'))
display(Image('results/colab_sgd/grad_error.png'))
display(Markdown(open('results/colab_sgd/summary.md').read()))

## LSTM language model (Penn Treebank)

Every LSTM matmul (`X @ W_ih` and `h @ W_hh` at every time step) and the output layer run in FP4
according to each configuration; the embedding (a table lookup) stays FP32. Language models are
scored by **perplexity** = exp(test loss): lower is better.

In [ ]:
import os, urllib.request
os.makedirs('data/ptb', exist_ok=True)
for split in ['train', 'test', 'valid']:
    path = f'data/ptb/{split}.txt'
    if not os.path.exists(path):
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.{split}.txt', path)
print({f: os.path.getsize(f'data/ptb/{f}') for f in os.listdir('data/ptb')})

In [ ]:
for seed in SEEDS:
    !python3 apps/fp4_lstm_experiment.py --configs all --device {DEVICE} --epochs {EPOCHS} --max-lines {LSTM_LINES} --seed {seed} --out results/colab_lstm_seed{seed}.csv
!python3 apps/plot_fp4.py results/colab_lstm_seed*.csv --out-dir results/colab_lstm --perplexity --title "PTB LSTM" --prefix lstm
display(Image('results/colab_lstm/lstm_grid.png'))
display(Image('results/colab_lstm/lstm_ablation.png'))
display(Markdown(open('results/colab_lstm/summary.md').read()))

## Optional: the MNIST grid with Adam
Adam adapts the step size per parameter, so it reacts differently to noisy or biased FP4 gradients.

In [ ]:
for seed in SEEDS:
    !python3 apps/fp4_experiment.py --configs all --device {DEVICE} --epochs {EPOCHS} --seed {seed} {SUBSET} --optimizer adam --lr 0.001 --out results/colab_adam_seed{seed}.csv
!python3 apps/plot_fp4.py results/colab_adam_seed*.csv --out-dir results/colab_adam
display(Image('results/colab_adam/mnist_grid.png'))
display(Image('results/colab_adam/mnist_ablation.png'))
display(Markdown(open('results/colab_adam/summary.md').read()))

Results (CSVs, plots, `summary.md`) are in `MyDrive/10714/fp4-needle/results/` on Drive.